In [ ]:
from pathlib import Path
from lexguard.utils.paths import RAW_DATA_DIR
from lexguard.ingestion.text_loader import load_text
from lexguard.ingestion.chunker import build_chunks

all_pages = []

for file in Path(RAW_DATA_DIR).glob("*.txt"):
    pages = load_text(str(file))
    all_pages.extend(pages)

chunks = build_chunks(
    document_id="multi_doc",
    title="Combined Policies",
    pages=all_pages
)

print("\nTotal chunks:", len(chunks))

for c in chunks:
    print("-" * 50)
    print("clause_id:", c.clause_id)
    print("section_id:", c.section_id)
    print("section_title:", c.section_title)
    print("text:", c.chunk_text)

In [ ]:
import json
from pathlib import Path

from lexguard.utils.paths import PROJECT_ROOT
from lexguard.retrieval.bm25 import BM25Retriever
from lexguard.retrieval.dense import DenseRetriever
from lexguard.retrieval.hybrid import HybridRetriever

benchmark_path = PROJECT_ROOT / "data" / "benchmarks" / "retrieval_smoke_test.json"

with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

bm25 = BM25Retriever(chunks)
dense = DenseRetriever(chunks)
hybrid = HybridRetriever(chunks, bm25_weight=0.5, dense_weight=0.5)

systems = {
    "bm25": bm25,
    "dense": dense,
    "hybrid": hybrid,
}

results = []

for item in benchmark:
    query = item["query"]
    expected = item["expected_section"]

    row = {
        "query": query,
        "expected": expected,
    }

    for name, system in systems.items():
        pred = system.query(query, top_k=1)[0].section_title
        row[name] = pred
        row[f"{name}_hit"] = int(pred == expected)

    results.append(row)

for row in results:
    print("=" * 80)
    print("QUERY:", row["query"])
    print("EXPECTED:", row["expected"])
    print("BM25   :", row["bm25"], "| hit =", row["bm25_hit"])
    print("DENSE  :", row["dense"], "| hit =", row["dense_hit"])
    print("HYBRID :", row["hybrid"], "| hit =", row["hybrid_hit"])

print("\nSummary")
for name in systems:
    score = sum(r[f"{name}_hit"] for r in results) / len(results)
    print(f"{name}: hit@1 = {score:.2f}")

In [ ]:
import json
from pathlib import Path

from lexguard.utils.paths import PROJECT_ROOT
from lexguard.retrieval.bm25 import BM25Retriever
from lexguard.retrieval.dense import DenseRetriever
from lexguard.retrieval.hybrid import HybridRetriever

benchmark_path = PROJECT_ROOT / "data" / "benchmarks" / "retrieval_smoke_test.json"

with open(benchmark_path, "r", encoding="utf-8") as f:
    benchmark = json.load(f)

bm25 = BM25Retriever(chunks)
dense = DenseRetriever(chunks)
hybrid = HybridRetriever(chunks, bm25_weight=0.5, dense_weight=0.5)

systems = {
    "bm25": bm25,
    "dense": dense,
    "hybrid": hybrid,
}

results = []

for item in benchmark:
    query = item["query"]
    expected = item["expected_section"]

    row = {
        "query": query,
        "expected": expected,
    }

    for name, system in systems.items():
        top1 = system.query(query, top_k=1)
        top2 = system.query(query, top_k=2)

        pred_top1 = top1[0].section_title
        pred_top2 = [r.section_title for r in top2]

        row[f"{name}_top1"] = pred_top1
        row[f"{name}_top2"] = pred_top2
        row[f"{name}_hit1"] = int(expected == pred_top1)
        row[f"{name}_hit2"] = int(expected in pred_top2)

    results.append(row)

for row in results:
    print("=" * 80)
    print("QUERY:", row["query"])
    print("EXPECTED:", row["expected"])
    for name in systems:
        print(
            f"{name.upper():7} top1={row[f'{name}_top1']!r} "
            f"| top2={row[f'{name}_top2']} "
            f"| hit@1={row[f'{name}_hit1']} "
            f"| hit@2={row[f'{name}_hit2']}"
        )

print("\nSummary")
for name in systems:
    hit1 = sum(r[f"{name}_hit1"] for r in results) / len(results)
    hit2 = sum(r[f"{name}_hit2"] for r in results) / len(results)
    print(f"{name}: hit@1 = {hit1:.2f}, hit@2 = {hit2:.2f}")

In [ ]:
from pathlib import Path

from lexguard.utils.paths import RAW_DATA_DIR
from lexguard.ingestion.pdf_loader import load_pdf
from lexguard.ingestion.chunker import build_chunks

cuad_dir = RAW_DATA_DIR / "cuad"
pdf_files = sorted(cuad_dir.glob("*.pdf"))

print("Total PDF files found:", len(pdf_files))
print("First 3 files:")
for f in pdf_files[:3]:
    print("-", f.name)

In [ ]:
from pathlib import Path

from lexguard.ingestion.pdf_loader import load_pdf

def load_cuad_pdf_folder(folder_path: str, limit: int | None = 5):
    folder = Path(folder_path)
    all_docs = []

    pdf_files = sorted(folder.glob("*.pdf"))
    if limit is not None:
        pdf_files = pdf_files[:limit]

    for file in pdf_files:
        pages = load_pdf(str(file))
        all_docs.append({
            "doc_id": file.stem,
            "title": file.stem,
            "pages": pages,
            "source_path": str(file),
        })

    return all_docs

In [ ]:
docs = load_cuad_pdf_folder(str(cuad_dir), limit=5)

all_chunks = []

for doc in docs:
    chunks = build_chunks(
        document_id=doc["doc_id"],
        title=doc["title"],
        pages=doc["pages"]
    )
    all_chunks.extend(chunks)

print("Documents loaded:", len(docs))
print("Total chunks:", len(all_chunks))

if all_chunks:
    print("\nFirst chunk example:")
    print("document_id:", all_chunks[0].document_id)
    print("section_title:", all_chunks[0].section_title)
    print("text:", all_chunks[0].chunk_text[:500])

In [ ]:
from lexguard.retrieval.bm25 import BM25Retriever
from lexguard.retrieval.hybrid import HybridRetriever

bm25 = BM25Retriever(all_chunks)
hybrid = HybridRetriever(all_chunks, bm25_weight=0.5, dense_weight=0.5)

In [ ]:
query = "termination for convenience"

results = hybrid.query(query, top_k=3)

for i, r in enumerate(results, start=1):
    print("=" * 80)
    print("Rank:", i)
    print("Document:", r.document_title)
    print("Section:", r.section_title)
    print("Text:", r.chunk_text[:800])

In [ ]:
docs = load_cuad_pdf_folder(str(cuad_dir), limit=3)

all_chunks = []

for doc in docs:
    chunks = build_chunks(
        document_id=doc["doc_id"],
        title=doc["title"],
        pages=doc["pages"]
    )
    all_chunks.extend(chunks)

for c in all_chunks[:10]:
    print("-" * 80)
    print("section_id:", c.section_id)
    print("section_title:", c.section_title)
    print("text:", c.chunk_text[:200])

In [ ]:
docs = load_cuad_pdf_folder(str(cuad_dir), limit=3)

all_chunks = []
for doc in docs:
    chunks = build_chunks(
        document_id=doc["doc_id"],
        title=doc["title"],
        pages=doc["pages"]
    )
    all_chunks.extend(chunks)

for c in all_chunks[:10]:
    print("-" * 80)
    print("section_id:", c.section_id)
    print("section_title:", c.section_title)
    print("text:", c.chunk_text[:250])

In [ ]:
docs = load_cuad_pdf_folder(str(cuad_dir), limit=3)

all_chunks = []
for doc in docs:
    chunks = build_chunks(
        document_id=doc["doc_id"],
        title=doc["title"],
        pages=doc["pages"]
    )
    all_chunks.extend(chunks)

for c in all_chunks[:10]:
    print("-" * 80)
    print("section_id:", c.section_id)
    print("section_title:", c.section_title)
    print("text:", c.chunk_text[:200])

In [ ]:
query = "governing law"

results = hybrid.query(query, top_k=5)

for r in results:
    print("="*80)
    print("doc:", r.document_title)
    print("section:", r.section_title)
    print("text:", r.chunk_text[:300])

In [ ]:
from pathlib import Path

from lexguard.utils.paths import RAW_DATA_DIR
from lexguard.ingestion.text_loader import load_text
from lexguard.ingestion.chunker import build_chunks

cuad_txt_dir = RAW_DATA_DIR / "cuad_txt"
txt_files = sorted(cuad_txt_dir.glob("*.txt"))

print("Total TXT files found:", len(txt_files))
for f in txt_files[:5]:
    print("-", f.name)

In [ ]:
from pathlib import Path

def load_cuad_txt_folder(folder_path: str, limit: int | None = 5):
    folder = Path(folder_path)
    all_docs = []

    txt_files = sorted(folder.glob("*.txt"))
    if limit is not None:
        txt_files = txt_files[:limit]

    for file in txt_files:
        pages = load_text(str(file))
        all_docs.append({
            "doc_id": file.stem,
            "title": file.stem,
            "pages": pages,
            "source_path": str(file),
        })

    return all_docs


docs = load_cuad_txt_folder(str(cuad_txt_dir), limit=5)

all_chunks = []

for doc in docs:
    chunks = build_chunks(
        document_id=doc["doc_id"],
        title=doc["title"],
        pages=doc["pages"]
    )
    all_chunks.extend(chunks)

print("Documents loaded:", len(docs))
print("Total chunks:", len(all_chunks))

for c in all_chunks[:5]:
    print("-" * 80)
    print("document_id:", c.document_id)
    print("section_id:", c.section_id)
    print("section_title:", c.section_title)
    print("text:", c.chunk_text[:300])

In [ ]:
docs = load_cuad_txt_folder(str(cuad_txt_dir), limit=5)

all_chunks = []

for doc in docs:
    chunks = build_chunks(
        document_id=doc["doc_id"],
        title=doc["title"],
        pages=doc["pages"]
    )
    all_chunks.extend(chunks)

print("Documents loaded:", len(docs))
print("Total chunks:", len(all_chunks))

for c in all_chunks[:10]:
    print("-" * 80)
    print("document_id:", c.document_id)
    print("section_id:", c.section_id)
    print("section_title:", c.section_title)
    print("text:", c.chunk_text[:300])

In [ ]:
docs = load_cuad_txt_folder(str(cuad_txt_dir), limit=5)

all_chunks = []

for doc in docs:
    chunks = build_chunks(
        document_id=doc["doc_id"],
        title=doc["title"],
        pages=doc["pages"]
    )
    all_chunks.extend(chunks)

print("Documents loaded:", len(docs))
print("Total chunks:", len(all_chunks))

for c in all_chunks[:5]:
    print("-" * 80)
    print("document_id:", c.document_id)
    print("section_id:", c.section_id)
    print("section_title:", c.section_title)
    print("text:", c.chunk_text[:300])

In [ ]:
from lexguard.retrieval.bm25 import BM25Retriever
from lexguard.retrieval.dense import DenseRetriever
from lexguard.retrieval.hybrid import HybridRetriever

bm25 = BM25Retriever(all_chunks)
dense = DenseRetriever(all_chunks)
hybrid = HybridRetriever(all_chunks, bm25_weight=0.5, dense_weight=0.5)

queries = [
    "governing law",
    "assignment",
    "termination",
    "confidential information",
    "audit rights",
    "exclusivity",
]

for q in queries:
    print("\n" + "=" * 100)
    print("QUERY:", q)
    results = hybrid.query(q, top_k=3)

    for i, r in enumerate(results, start=1):
        print("-" * 80)
        print("rank:", i)
        print("doc:", r.document_title)
        print("section_id:", r.section_id)
        print("section_title:", r.section_title)
        print("text:", r.chunk_text[:500])

In [ ]:
test_queries = [
    "governing law",
    "assignment",
    "exclusivity",
]

for q in test_queries:
    print("\n" + "=" * 100)
    print("QUERY:", q)

    bm25_results = bm25.query(q, top_k=3)
    hybrid_results = hybrid.query(q, top_k=3)

    print("\nBM25")
    for i, r in enumerate(bm25_results, start=1):
        print("-" * 80)
        print("rank:", i)
        print("doc:", r.document_title)
        print("section_id:", r.section_id)
        print("section_title:", r.section_title)
        print("text:", r.chunk_text[:350])

    print("\nHYBRID")
    for i, r in enumerate(hybrid_results, start=1):
        print("-" * 80)
        print("rank:", i)
        print("doc:", r.document_title)
        print("section_id:", r.section_id)
        print("section_title:", r.section_title)
        print("text:", r.chunk_text[:350])